In [2]:
!pip install qiskit qiskit-aer qiskit-machine-learning torch torchvision kagglehub scikit-learn seaborn grad-cam pydicom --quiet

import os, numpy as np, torch
import torch.nn as nn, torch.optim as optim
from torch.utils.data import DataLoader, WeightedRandomSampler, ConcatDataset, Dataset
from torchvision import datasets, models, transforms
from qiskit.circuit import QuantumCircuit, ParameterVector
from qiskit.quantum_info import SparsePauliOp
from qiskit_machine_learning.neural_networks import EstimatorQNN
from qiskit_machine_learning.connectors import TorchConnector
from sklearn.metrics import classification_report, confusion_matrix
import matplotlib.pyplot as plt, seaborn as sns, kagglehub
import warnings; warnings.filterwarnings("ignore")
from pytorch_grad_cam import GradCAM
from pytorch_grad_cam.utils.image import show_cam_on_image
from pytorch_grad_cam.utils.model_targets import ClassifierOutputTarget
import pydicom
from PIL import Image
from collections import Counter

# ── Device auto-detect (works on Kaggle, Colab, and local NVIDIA GPU) ─
if torch.cuda.is_available():
    DEVICE = torch.device("cuda")
    print(f"✅ GPU detected: {torch.cuda.get_device_name(0)}")
else:
    DEVICE = torch.device("cpu")
    print("⚠️  No CUDA GPU found — running on CPU (slow but works)")

N_QUBITS = 4
BATCH    = 16        # reduce to 8 if you get OOM on local GPU
SEED     = 42

# ── Save directory: works on Kaggle, Colab, and local ─────────────────
if os.path.exists("/kaggle/working"):
    SAVE_DIR = "/kaggle/working"
elif os.path.exists("/content"):
    SAVE_DIR = "/content/qcnn_checkpoints"   # Google Colab
else:
    SAVE_DIR = os.path.join(os.path.expanduser("~"), "qcnn_checkpoints")  # local

os.makedirs(SAVE_DIR, exist_ok=True)
torch.manual_seed(SEED); np.random.seed(SEED)
print(f"Device   : {DEVICE}")
print(f"Qubits   : {N_QUBITS}")
print(f"Save dir : {SAVE_DIR}")

c:\Users\rames\miniconda3\envs\qtb\lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


⚠️  No CUDA GPU found — running on CPU (slow but works)
Device   : cpu
Qubits   : 4
Save dir : C:\Users\rames\qcnn_checkpoints


In [3]:
if os.path.exists("/kaggle/input/datasets/sowmyaabirami/icmr-tuberclosis-dataset/intbtr254"):
    ICMR_ROOT = "/kaggle/input/datasets/sowmyaabirami/icmr-tuberclosis-dataset/intbtr254"
elif os.path.exists("/content/intbtr254"):
    ICMR_ROOT = "/content/intbtr254"
elif os.path.exists(r"C:\Users\rames\OneDrive\Documents\All coding projects\QML Tuberclosis project\intbtr254"):
    ICMR_ROOT = r"C:\Users\rames\OneDrive\Documents\All coding projects\QML Tuberclosis project\intbtr254"
else:
    ICMR_ROOT = os.path.join(os.path.expanduser("~"), "datasets", "intbtr254")

print(f"ICMR path : {ICMR_ROOT}")
print(f"Exists    : {os.path.exists(ICMR_ROOT)}")

# ── Download original datasets ────────────────────────────────────────
path1 = kagglehub.dataset_download("tawsifurrahman/tuberculosis-tb-chest-xray-dataset")
path2 = kagglehub.dataset_download("tirthachhetri/tuberclosis-dataset")
print("DB1:", path1, "\nDB2:", path2)

# ── Path Discovery ────────────────────────────────────────────────────
def find_dir(root, target):
    for dirpath, dirnames, _ in os.walk(root):
        if os.path.basename(dirpath) == target:
            return dirpath
    return root

db1_root = find_dir(path1, "TB_Chest_Radiography_Database")
tb_train = find_dir(path2, "train")
tb_val   = find_dir(path2, "valid")
tb_test  = find_dir(path2, "test")

print(f"\nDB1 root : {db1_root}")
print(f"TB_split : train={tb_train} | val={tb_val} | test={tb_test}")

# ── Transforms ───────────────────────────────────────────────────────
train_tf = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.Grayscale(num_output_channels=3),
    transforms.RandomHorizontalFlip(),
    transforms.RandomRotation(10),
    transforms.ColorJitter(brightness=0.2, contrast=0.2),
    transforms.ToTensor(),
    transforms.Normalize([0.485, 0.456, 0.406], [0.229, 0.224, 0.225])
])
val_tf = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.Grayscale(num_output_channels=3),
    transforms.ToTensor(),
    transforms.Normalize([0.485, 0.456, 0.406], [0.229, 0.224, 0.225])
])

# ── DICOM wrapper for ICMR dataset ───────────────────────────────────
class DICOMDataset(Dataset):
    """
    All 162 ICMR files are Abnormal (class index 1).
    Pixel array → normalized uint8 → PIL RGB to match ImageFolder pipeline.
    """
    def __init__(self, folder, transform=None):
        self.files = sorted([
            os.path.join(folder, f)
            for f in os.listdir(folder)
            if f.lower().endswith(".dicom")
        ])
        self.transform = transform

    def __len__(self):
        return len(self.files)

    def __getitem__(self, idx):
        dcm = pydicom.dcmread(self.files[idx])
        arr = dcm.pixel_array.astype(np.float32)
        arr = (arr - arr.min()) / (arr.max() - arr.min() + 1e-8) * 255
        img = Image.fromarray(arr.astype(np.uint8)).convert("RGB")
        if self.transform:
            img = self.transform(img)
        return img, 1   # hard-label: Abnormal = 1

icmr_ds = DICOMDataset(ICMR_ROOT, transform=train_tf)

# ── ImageFolder datasets ──────────────────────────────────────────────
extra_ds    = datasets.ImageFolder(db1_root, transform=train_tf)
split_train = datasets.ImageFolder(tb_train, transform=train_tf)
val_ds      = datasets.ImageFolder(tb_val,   transform=val_tf)
test_ds     = datasets.ImageFolder(tb_test,  transform=val_tf)

# ── Merge all three into training set ────────────────────────────────
train_ds = ConcatDataset([split_train, extra_ds, icmr_ds])
classes  = split_train.classes
print(f"\nClasses : {classes}")

# ── Label counts per source (fast — no image loading) ─────────────────
split_labels = [label for _, label in split_train.samples]
extra_labels = [label for _, label in extra_ds.samples]
icmr_labels  = [1] * len(icmr_ds)
all_labels   = split_labels + extra_labels + icmr_labels

split_counts = Counter(split_labels)
extra_counts = Counter(extra_labels)
icmr_counts  = Counter(icmr_labels)
total_counts = Counter(all_labels)

print(f"\n{'─'*52}")
print(f"{'Source':<20} {'Normal':>10} {'TB/Abnormal':>12} {'Total':>7}")
print(f"{'─'*52}")
print(f"{'tirthachhetri train':<20} {split_counts[0]:>10} {split_counts[1]:>12} {len(split_train):>7}")
print(f"{'tawsifurrahman DB1':<20} {extra_counts[0]:>10} {extra_counts[1]:>12} {len(extra_ds):>7}")
print(f"{'ICMR DICOM':<20} {icmr_counts[0]:>10} {icmr_counts[1]:>12} {len(icmr_ds):>7}")
print(f"{'─'*52}")
print(f"{'TOTAL TRAIN':<20} {total_counts[0]:>10} {total_counts[1]:>12} {len(all_labels):>7}")
print(f"{'─'*52}")
print(f"\nVal   : {len(val_ds):,} images")
print(f"Test  : {len(test_ds):,} images")

# ── Weighted Sampler ──────────────────────────────────────────────────
counts_arr = np.bincount(all_labels)
weights    = (1.0 / np.array(counts_arr)[all_labels]).tolist()
sampler    = WeightedRandomSampler(weights, len(weights))

# num_workers=0 for local Windows machines (avoids multiprocessing issues)
nw = 0 if os.name == 'nt' else 2

train_loader = DataLoader(train_ds, batch_size=BATCH, sampler=sampler, num_workers=nw)
val_loader   = DataLoader(val_ds,   batch_size=BATCH, shuffle=False,   num_workers=nw)
test_loader  = DataLoader(test_ds,  batch_size=BATCH, shuffle=False,   num_workers=nw)

print(f"\n{'─'*52}")
print(f"{'Loader':<15} {'Images':>8} {'Batches (size='+str(BATCH)+')':>20}")
print(f"{'─'*52}")
print(f"{'train_loader':<15} {len(train_ds):>8} {len(train_loader):>20}")
print(f"{'val_loader':<15} {len(val_ds):>8} {len(val_loader):>20}")
print(f"{'test_loader':<15} {len(test_ds):>8} {len(test_loader):>20}")
print(f"{'─'*52}")
print("\nDataLoaders ready.")

ICMR path : C:\Users\rames\OneDrive\Documents\All coding projects\QML Tuberclosis project\intbtr254
Exists    : True
DB1: C:\Users\rames\.cache\kagglehub\datasets\tawsifurrahman\tuberculosis-tb-chest-xray-dataset\versions\3 
DB2: C:\Users\rames\.cache\kagglehub\datasets\tirthachhetri\tuberclosis-dataset\versions\1

DB1 root : C:\Users\rames\.cache\kagglehub\datasets\tawsifurrahman\tuberculosis-tb-chest-xray-dataset\versions\3\TB_Chest_Radiography_Database
TB_split : train=C:\Users\rames\.cache\kagglehub\datasets\tirthachhetri\tuberclosis-dataset\versions\1\TB_split\train | val=C:\Users\rames\.cache\kagglehub\datasets\tirthachhetri\tuberclosis-dataset\versions\1\TB_split\valid | test=C:\Users\rames\.cache\kagglehub\datasets\tirthachhetri\tuberclosis-dataset\versions\1\TB_split\test

Classes : ['healthy', 'tb']

────────────────────────────────────────────────────
Source                   Normal  TB/Abnormal   Total
────────────────────────────────────────────────────
tirthachhetri train

In [4]:
def build_qcnn_circuit(n_qubits=4):
    qc       = QuantumCircuit(n_qubits)
    x_params = ParameterVector('x', n_qubits)
    h_params = ParameterVector('h', 8)
    m_params = ParameterVector('m', 6)
    b_params = ParameterVector('b', 2)

    for i in range(n_qubits):
        qc.ry(x_params[i], i)
    qc.barrier()

    qc.ry(h_params[0], 0); qc.ry(h_params[1], 1)
    qc.cx(0, 1)
    qc.ry(h_params[2], 0); qc.ry(h_params[3], 1)
    qc.ry(h_params[4], 2); qc.ry(h_params[5], 3)
    qc.cx(2, 3)
    qc.ry(h_params[6], 2); qc.ry(h_params[7], 3)
    qc.barrier()

    qc.crz(m_params[0], 0, 1)
    qc.crz(m_params[1], 2, 3)
    qc.ry(m_params[2], 1); qc.ry(m_params[3], 3)
    qc.cx(1, 3)
    qc.ry(m_params[4], 1); qc.ry(m_params[5], 3)
    qc.barrier()

    qc.crz(b_params[0], 1, 3)
    qc.ry(b_params[1], 3)

    return qc, x_params, h_params, m_params, b_params


class HybridQCNN(nn.Module):
    def __init__(self, n_qubits=N_QUBITS):
        super().__init__()

        res = models.resnet18(weights="IMAGENET1K_V1")
        for p in res.parameters(): p.requires_grad = False
        self.backbone  = nn.Sequential(*list(res.children())[:-1])
        self.projector = nn.Sequential(
            nn.Flatten(),
            nn.Linear(512, n_qubits),
            nn.Tanh()
        )

        qc, x_p, h_p, m_p, b_p = build_qcnn_circuit(n_qubits)
        obs = SparsePauliOp("ZIII")

        self.qnn = EstimatorQNN(
            circuit         = qc,
            input_params    = list(x_p),
            weight_params   = list(h_p) + list(m_p) + list(b_p),
            observables     = obs,
            input_gradients = True,
        )

        init_w         = np.random.uniform(-np.pi, np.pi, self.qnn.num_weights)
        self.connector = TorchConnector(self.qnn, initial_weights=init_w)

        self._hs   = len(h_p)
        self._ms   = len(m_p)
        self._bs   = len(b_p)
        self._mask = torch.zeros(self.qnn.num_weights)
        self._mask[:self._hs] = 1
        self.connector.weight.register_hook(
            lambda grad: grad * self._mask.to(grad.device)
        )

        self.out_head = nn.Linear(1, 2)
        self._print_layers()

    def set_phase(self, phase: int):
        mask = torch.zeros(self.qnn.num_weights)
        hs, ms = self._hs, self._ms
        if phase >= 1: mask[:hs]      = 1
        if phase >= 2: mask[hs:hs+ms] = 1
        if phase >= 3: mask[hs+ms:]   = 1
        self._mask = mask
        labels = {1: "HEAD only", 2: "HEAD + MIDDLE", 3: "HEAD + MIDDLE + BODY (full)"}
        print(f"\n{'─'*56}")
        print(f" Phase {phase} unlocked → {labels[phase]}")
        print(f" Active quantum params : {int(mask.sum())}/{self.qnn.num_weights}")
        print(f"{'─'*56}")

    def _print_layers(self):
        print("\n╔══ QCNN Architecture ══════════════════════════════════╗")
        rows = [
            ("1 (HEAD)",   "Conv Layer 1 — 2-qubit unitaries (q0-q1, q2-q3)", self._hs),
            ("2 (MIDDLE)", "Pool1 (CRZ) + Conv Layer 2 — active qubits q1,q3", self._ms),
            ("3 (BODY)",   "Pool2 (CRZ) + Final rotation — output qubit q3",   self._bs),
        ]
        for lyr, desc, params in rows:
            print(f"║  Layer {lyr:<10} │ {desc:<40} │ {params} params ║")
        print(f"╠═══════════════════════════════════════════════════════╣")
        total_q  = self.qnn.num_weights
        total_cl = sum(p.numel() for p in self.parameters()) - total_q
        print(f"║  Total QCNN layers    : 3                             ║")
        print(f"║  Total quantum params : {total_q:<31}║")
        print(f"║  Circuit depth        : {self.qnn.circuit.depth():<31}║")
        print(f"║  Qubits               : {N_QUBITS:<31}║")
        print(f"║  Classical params     : {total_cl:<31,}║")
        print(f"╚═══════════════════════════════════════════════════════╝\n")

    def forward(self, x):
        x = self.backbone(x)
        x = self.projector(x) * np.pi
        x = self.connector(x)
        return self.out_head(x)

model = HybridQCNN().to(DEVICE)

No gradient function provided, creating a gradient function. If your Estimator requires transpilation, please provide a pass manager.



╔══ QCNN Architecture ══════════════════════════════════╗
║  Layer 1 (HEAD)   │ Conv Layer 1 — 2-qubit unitaries (q0-q1, q2-q3) │ 8 params ║
║  Layer 2 (MIDDLE) │ Pool1 (CRZ) + Conv Layer 2 — active qubits q1,q3 │ 6 params ║
║  Layer 3 (BODY)   │ Pool2 (CRZ) + Final rotation — output qubit q3 │ 2 params ║
╠═══════════════════════════════════════════════════════╣
║  Total QCNN layers    : 3                             ║
║  Total quantum params : 16                             ║
║  Circuit depth        : 10                             ║
║  Qubits               : 4                              ║
║  Classical params     : 11,178,568                     ║
╚═══════════════════════════════════════════════════════╝



In [5]:
PHASES = {
    1: dict(epochs=5, lr=1e-3,  label="HEAD only"),
    2: dict(epochs=5, lr=5e-4,  label="HEAD + MIDDLE"),
    3: dict(epochs=5, lr=2e-4,  label="Full QCNN"),
}
criterion = nn.CrossEntropyLoss()
history   = {"phase":[], "epoch":[], "train_loss":[], "val_acc":[]}
best_val  = {1: 0.0, 2: 0.0, 3: 0.0}

# ── Resume Detection ──────────────────────────────────────────────────
completed_phases = []
resume_phase     = None
resume_ckpt_path = None

for p in [1, 2, 3]:
    final = os.path.join(SAVE_DIR, f"qcnn_phase{p}_final.pth")
    best  = os.path.join(SAVE_DIR, f"qcnn_phase{p}_best.pth")
    if os.path.exists(final):
        completed_phases.append(p)
    elif os.path.exists(best) and p not in completed_phases:
        resume_phase     = p
        resume_ckpt_path = best
        break

print(f"\n{'═'*60}")
print(f"  📂 Scanning {SAVE_DIR} for existing checkpoints...")
for f in sorted(os.listdir(SAVE_DIR)):
    if f.endswith(".pth"):
        fsize = os.path.getsize(os.path.join(SAVE_DIR, f)) / 1e6
        print(f"     ✓ Found: {f:<40} {fsize:.2f} MB")

if completed_phases:
    print(f"\n  ✅ Fully completed phases : {completed_phases}")
if resume_phase:
    print(f"  ⚠️  Phase {resume_phase} was interrupted — resuming from best checkpoint")
if not completed_phases and not resume_phase:
    print(f"\n  🆕 No checkpoints found — starting fresh from Phase 1")
print(f"{'═'*60}")

# ── Load weights + history if resuming ───────────────────────────────
if completed_phases or resume_phase:
    load_path = resume_ckpt_path if resume_phase else os.path.join(
        SAVE_DIR, f"qcnn_phase{max(completed_phases)}_final.pth"
    )
    ckpt = torch.load(load_path, map_location=DEVICE)
    model.load_state_dict(ckpt['model_state_dict'])
    history  = ckpt.get('history', history)
    best_val = {p: max(
        [v for ph, v in zip(history['phase'], history['val_acc']) if ph == p],
        default=0.0
    ) for p in [1, 2, 3]}
    print(f"\n  ✅ Loaded weights from  : {load_path}")
    print(f"  📈 Best val accs so far : { {k: f'{v:.2f}%' for k,v in best_val.items()} }")

# ── Training Loop ─────────────────────────────────────────────────────
for phase, cfg in PHASES.items():

    if phase in completed_phases:
        print(f"\n  ⏭️  Phase {phase} already complete — skipping")
        continue

    model.set_phase(phase)
    optimizer = optim.Adam(model.parameters(), lr=cfg['lr'])
    scheduler = optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=cfg['epochs'])

    if phase == resume_phase and resume_ckpt_path:
        ckpt = torch.load(resume_ckpt_path, map_location=DEVICE)
        if 'optimizer_state_dict' in ckpt:
            optimizer.load_state_dict(ckpt['optimizer_state_dict'])
        print(f"\n  ♻️  Optimizer state restored for Phase {phase}")

    print(f"\n{'═'*60}")
    print(f"  Phase {phase} | {cfg['label']}  —  {cfg['epochs']} epochs")
    print(f"{'═'*60}")

    for epoch in range(1, cfg['epochs'] + 1):

        model.train(); total_loss = 0.0
        for imgs, labels in train_loader:
            imgs, labels = imgs.to(DEVICE), labels.to(DEVICE)
            optimizer.zero_grad()
            loss = criterion(model(imgs), labels)
            loss.backward(); optimizer.step()
            total_loss += loss.item()

        model.eval(); correct = total = 0
        with torch.no_grad():
            for imgs, labels in val_loader:
                preds   = model(imgs.to(DEVICE)).argmax(1).cpu()
                correct += (preds == labels).sum().item()
                total   += len(labels)

        avg_loss = total_loss / len(train_loader)
        val_acc  = 100.0 * correct / total
        history["phase"].append(phase)
        history["epoch"].append(epoch)
        history["train_loss"].append(avg_loss)
        history["val_acc"].append(val_acc)
        scheduler.step()

        print(f"  Epoch {epoch:02d}/{cfg['epochs']} │ "
              f"Loss: {avg_loss:.4f} │ Val Acc: {val_acc:.2f}%", end="")

        if val_acc > best_val[phase]:
            best_val[phase] = val_acc
            ckpt_path = os.path.join(SAVE_DIR, f"qcnn_phase{phase}_best.pth")
            torch.save({
                'phase'               : phase,
                'epoch'               : epoch,
                'model_state_dict'    : model.state_dict(),
                'optimizer_state_dict': optimizer.state_dict(),
                'val_acc'             : val_acc,
                'history'             : history,
                'classes'             : classes,
            }, ckpt_path)
            print(f"  ← ✓ best saved ({val_acc:.2f}%)", end="")
        print()

    final_ckpt = os.path.join(SAVE_DIR, f"qcnn_phase{phase}_final.pth")
    torch.save({
        'phase'               : phase,
        'model_state_dict'    : model.state_dict(),
        'optimizer_state_dict': optimizer.state_dict(),
        'val_acc'             : val_acc,
        'history'             : history,
        'classes'             : classes,
    }, final_ckpt)
    print(f"\n  📦 Phase {phase} final → {final_ckpt}")
    print(f"  🏆 Phase {phase} best  → {SAVE_DIR}/qcnn_phase{phase}_best.pth")

# ── Save full trained model ───────────────────────────────────────────
full_path = os.path.join(SAVE_DIR, "qcnn_full_trained.pth")
torch.save({
    'model_state_dict': model.state_dict(),
    'history'         : history,
    'classes'         : classes,
    'n_qubits'        : N_QUBITS,
}, full_path)

total_epochs = sum(c['epochs'] for c in PHASES.values())
print(f"\n{'═'*60}")
print(f"  ✓  Progressive training complete")
print(f"     Total epochs : {total_epochs}  |  Phases : 3 / 3")
print(f"     Path : HEAD → HEAD+MIDDLE → FULL QCNN")
print(f"{'═'*60}")
print(f"\n  📁 All files in {SAVE_DIR}:")
for f in sorted(os.listdir(SAVE_DIR)):
    if f.endswith(".pth"):
        fsize = os.path.getsize(os.path.join(SAVE_DIR, f)) / 1e6
        print(f"     {f:<40} {fsize:.2f} MB")


════════════════════════════════════════════════════════════
  📂 Scanning C:\Users\rames\qcnn_checkpoints for existing checkpoints...
     ✓ Found: qcnn_phase1_best.pth                     44.82 MB
     ✓ Found: qcnn_phase1_final.pth                    44.82 MB
     ✓ Found: qcnn_phase2_best.pth                     44.82 MB
     ✓ Found: qcnn_phase2_final.pth                    44.82 MB
     ✓ Found: qcnn_phase3_best.pth                     44.82 MB

  ✅ Fully completed phases : [1, 2]
  ⚠️  Phase 3 was interrupted — resuming from best checkpoint
════════════════════════════════════════════════════════════

  ✅ Loaded weights from  : C:\Users\rames\qcnn_checkpoints\qcnn_phase3_best.pth
  📈 Best val accs so far : {1: '89.52%', 2: '90.61%', 3: '88.66%'}

  ⏭️  Phase 1 already complete — skipping

  ⏭️  Phase 2 already complete — skipping

────────────────────────────────────────────────────────
 Phase 3 unlocked → HEAD + MIDDLE + BODY (full)
 Active quantum params : 16/16
──────────────

KeyboardInterrupt: 

In [ ]:
# ── Test Set Evaluation ───────────────────────────────────────────────
model.eval()
all_preds, all_true = [], []
with torch.no_grad():
    for imgs, labels in test_loader:
        preds = model(imgs.to(DEVICE)).argmax(1).cpu().numpy()
        all_preds.extend(preds); all_true.extend(labels.numpy())

print("\n── Test Classification Report ──────────────────────────────")
print(classification_report(all_true, all_preds, target_names=test_ds.classes))

# ── Figure 1: Training Curves + Confusion Matrix ──────────────────────
fig1, axes = plt.subplots(1, 3, figsize=(18, 5))
fig1.suptitle("Hybrid QCNN — TB Chest X-Ray Results", fontsize=14, fontweight='bold')
phase_colors = {1: "coral", 2: "steelblue", 3: "seagreen"}

cm = confusion_matrix(all_true, all_preds)
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues',
            xticklabels=test_ds.classes, yticklabels=test_ds.classes,
            ax=axes[0], linewidths=0.5)
axes[0].set_title("Confusion Matrix")
axes[0].set_ylabel("True"); axes[0].set_xlabel("Predicted")

for ph in [1, 2, 3]:
    idx    = [i for i, p in enumerate(history['phase']) if p == ph]
    epochs = [history['epoch'][i]      for i in idx]
    losses = [history['train_loss'][i] for i in idx]
    axes[1].plot(epochs, losses, color=phase_colors[ph], marker='o',
                 label=f'Phase {ph}: {PHASES[ph]["label"]}')
axes[1].set_title("Train Loss per Phase")
axes[1].set_xlabel("Epoch within Phase"); axes[1].legend(fontsize=8)

for ph in [1, 2, 3]:
    idx    = [i for i, p in enumerate(history['phase']) if p == ph]
    epochs = [history['epoch'][i]   for i in idx]
    accs   = [history['val_acc'][i] for i in idx]
    axes[2].plot(epochs, accs, color=phase_colors[ph], marker='s',
                 label=f'Phase {ph}: {PHASES[ph]["label"]}')
axes[2].set_title("Val Accuracy per Phase")
axes[2].set_xlabel("Epoch within Phase"); axes[2].set_ylabel("Accuracy (%)")
axes[2].legend(fontsize=8)

plt.tight_layout()
curve_path = os.path.join(SAVE_DIR, "qcnn_training_curves.png")
plt.savefig(curve_path, dpi=150, bbox_inches='tight')
plt.show()
print(f"📦 Saved → {curve_path}")

# ── Figure 2: Grad-CAM ────────────────────────────────────────────────
target_layer = [model.backbone[7]]
for p in model.backbone.parameters():
    p.requires_grad_(True)

cam_engine = GradCAM(model=model, target_layers=target_layer)

correct_imgs, correct_labels, correct_preds = [], [], []
wrong_imgs,   wrong_labels,   wrong_preds   = [], [], []

with torch.no_grad():
    for imgs, labels in test_loader:
        preds_b = model(imgs.to(DEVICE)).argmax(1).cpu()
        for i in range(len(labels)):
            if preds_b[i] == labels[i] and len(correct_imgs) < 4:
                correct_imgs.append(imgs[i]); correct_labels.append(labels[i].item())
                correct_preds.append(preds_b[i].item())
            elif preds_b[i] != labels[i] and len(wrong_imgs) < 4:
                wrong_imgs.append(imgs[i]); wrong_labels.append(labels[i].item())
                wrong_preds.append(preds_b[i].item())
        if len(correct_imgs) == 4 and len(wrong_imgs) == 4:
            break

MEAN = np.array([0.485, 0.456, 0.406])
STD  = np.array([0.229, 0.224, 0.225])

def denorm(t):
    img = t.numpy().transpose(1, 2, 0)
    img = img * STD + MEAN
    return np.clip(img, 0, 1).astype(np.float32)

def get_cam_overlays(img_tensors, labels, preds):
    overlays = []
    for img_t, lbl, pred in zip(img_tensors, labels, preds):
        inp     = img_t.unsqueeze(0)
        target  = [ClassifierOutputTarget(pred)]
        mask    = cam_engine(input_tensor=inp, targets=target)
        rgb     = denorm(img_t)
        overlay = show_cam_on_image(rgb, mask[0], use_rgb=True, image_weight=0.55)
        overlays.append((overlay, lbl, pred))
    return overlays

correct_cams = get_cam_overlays(correct_imgs, correct_labels, correct_preds)
wrong_cams   = get_cam_overlays(wrong_imgs,   wrong_labels,   wrong_preds)

n_cols = max(len(correct_cams), len(wrong_cams))
fig2, axes2 = plt.subplots(2, n_cols, figsize=(n_cols * 4, 9))
fig2.suptitle("Grad-CAM — ResNet18 Layer4 Activations\n"
              "(Row 1: Correct  |  Row 2: Wrong)",
              fontsize=13, fontweight='bold')

for row_idx, (cams, row_label, color) in enumerate([
    (correct_cams, "✓ Correct", "green"),
    (wrong_cams,   "✗ Wrong",   "crimson")
]):
    for col_idx in range(n_cols):
        ax = axes2[row_idx, col_idx]
        if col_idx < len(cams):
            overlay, true_lbl, pred_lbl = cams[col_idx]
            ax.imshow(overlay)
            ax.set_title(
                f"True: {test_ds.classes[true_lbl]}\nPred: {test_ds.classes[pred_lbl]}",
                fontsize=9, color=color)
            for spine in ax.spines.values():
                spine.set_edgecolor(color); spine.set_linewidth(3)
        else:
            ax.axis('off')
        ax.set_xticks([]); ax.set_yticks([])
    axes2[row_idx, 0].set_ylabel(row_label, fontsize=12,
                                  color=color, fontweight='bold')

plt.tight_layout()
cam_path = os.path.join(SAVE_DIR, "qcnn_gradcam.png")
plt.savefig(cam_path, dpi=150, bbox_inches='tight')
plt.show()
print(f"📦 Saved → {cam_path}")

for p in model.backbone.parameters():
    p.requires_grad_(False)

total_params = sum(p.numel() for p in model.parameters())
print("\n╔══ Final QCNN Summary ══════════════════════════════════════╗")
print(f"║  Quantum circuit depth    : {model.qnn.circuit.depth():<28}║")
print(f"║  Qubits                   : {N_QUBITS:<28}║")
print(f"║  QCNN layers (groups)     : 3  (HEAD | MIDDLE | BODY)    ║")
print(f"║  Total quantum params     : {model.qnn.num_weights:<28}║")
print(f"║  Total model params       : {total_params:<28,}║")
print(f"║  Training phases done     : 3                            ║")
print(f"╚════════════════════════════════════════════════════════════╝")

print(f"\n  📁 All files in {SAVE_DIR}:")
for f in sorted(os.listdir(SAVE_DIR)):
    fsize = os.path.getsize(os.path.join(SAVE_DIR, f)) / 1e6
    print(f"     {f:<45} {fsize:.2f} MB")

In [ ]:
model.eval()
dummy_input = torch.randn(1, 3, 224, 224)

with torch.no_grad():
    traced_model = torch.jit.trace(model, dummy_input)

export_path = os.path.join(SAVE_DIR, "qtb_traced.pt")
torch.jit.save(traced_model, export_path)

reloaded = torch.jit.load(export_path)
reloaded.eval()
out = reloaded(dummy_input)
print(f"✓ Export successful → {export_path}")
print(f"  Output shape : {out.shape}")
print(f"  Test logits  : {out}")